# Market Regime Detection for Portfolio Allocation with RL + Explainability

This notebook builds a regime-aware portfolio allocation pipeline and then explains policy behavior with temporal attention-style diagnostics.

## What This Notebook Covers
1. Regime detection with Gaussian HMM
2. RL training with DQN, A2C, and PPO
3. Test-period evaluation and policy behavior diagnostics
4. Explainability views:
   - DQN surrogate attention + gradient saliency
   - Finance dashboard linking returns, actions, and regimes
   - PPO post-hoc surrogate attention + policy saliency

## Important Clarification
- DQN section uses value-based learning and is explained with surrogate attention + saliency.
- PPO section in this notebook uses `MlpPolicy` (no native attention layer in PPO policy).
- PPO attention plots are post-hoc explainability, not internal PPO attention weights.

## Execution Guide (Recommended Order)

1. Run Sections 1 to 5 to prepare data and environments.
2. Choose training path:
   - DQN path: Section 6 + Sections 7 to 9.2
   - PPO path: PPO training cell + Section 9.3
3. Run Section 10 only when comparing against the baseline setup.

This order keeps variables aligned and avoids stale-state confusion in long notebook sessions.

## 1. Install and Import Dependencies

In [1]:
import os
import sys
from pathlib import Path

repo_root = Path.cwd().resolve()
if not (repo_root / "ml").exists():
    found_root = None
    for candidate in [repo_root] + list(repo_root.parents):
        if (candidate / "ml").exists() and (candidate / "data").exists():
            found_root = candidate
            break
    if found_root is None:
        raise FileNotFoundError("Could not locate project root containing 'ml' and 'data'.")
    repo_root = found_root

sys.path.insert(0, str(repo_root))

def _rel_display(path_obj: Path) -> str:
    try:
        return str(path_obj.relative_to(repo_root))
    except ValueError:
        return str(path_obj)

import numpy as np
import pandas as pd
import seaborn as sns
import warnings

# FinRL (stable-baselines3)
from stable_baselines3 import DQN, A2C, PPO
from stable_baselines3.common.vec_env import DummyVecEnv

# ML libraries
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Project modules
from ml.models import GaussianHMMRegimeDetector
from ml.environments import WeeklyPortfolioEnv
from ml.training_utils import train_dqn_finrl, evaluate_episode
from ml import load_hyperparameter_config

# Visualization
from tqdm import tqdm

warnings.filterwarnings('ignore')

# Load centralized hyperparameters
hp_bundle = load_hyperparameter_config()
hp_cfg = hp_bundle.values
FAST_MODE = hp_bundle.fast_mode

env_cfg = hp_cfg['environment']
dqn_cfg = hp_cfg['dqn']
a2c_cfg = hp_cfg['a2c']
ppo_cfg = hp_cfg['ppo']
ppo_attn_cfg = hp_cfg['ppo_native_attention']
hpo_cfg = hp_cfg['hpo']
SEQ_LEN = hp_cfg['general']['sequence_length']

print(f"Hyperparameter mode: {'FAST' if FAST_MODE else 'FULL'}")
print(f"Configured sequence length: {SEQ_LEN}")

# Saved model artifacts for split-notebook workflow
model_dir = repo_root / 'output' / 'models'
MODEL_PATHS = {
    'dqn': model_dir / 'dqn_finrl_agent.zip',
    'a2c': model_dir / 'a2c_finrl_agent.zip',
    'ppo': model_dir / 'ppo_finrl_agent.zip',
    'ppo_native_attention': model_dir / 'ppo_native_attention_finrl_agent.zip',
}
print(f"Model artifacts directory: {_rel_display(model_dir)}")

# Set random seeds for reproducibility
np.random.seed(42)
import torch
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)
elif torch.backends.mps.is_available():
    torch.mps.manual_seed(42)

# Device configuration (order: CUDA > MPS > CPU)
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
print(f"Using device: {device}")

sns.set_palette("husl")

Hyperparameter mode: FULL
Configured sequence length: 4
Model artifacts directory: output/models
Using device: cuda


## 2. Load and Prepare Data from Pipeline

In [2]:
# Load processed data from the data pipeline
# Use repo_root so this works regardless of current working directory.
data_path = repo_root / 'data' / 'processed'

# Load weekly features (market + macro)
market_features = pd.read_csv(data_path / 'market_features_weekly.csv', index_col=0, parse_dates=True)
macro_features = pd.read_csv(data_path / 'macro_features_weekly.csv', index_col=0, parse_dates=True)
asset_targets = pd.read_csv(data_path / 'weekly_asset_targets.csv', index_col=0, parse_dates=True)

print("Data path: data/processed")
print(f"Market features shape: {market_features.shape}")
print(f"Macro features shape: {macro_features.shape}")
print(f"Asset targets shape: {asset_targets.shape}")

# Align indices
common_index = market_features.index.intersection(macro_features.index).intersection(asset_targets.index)
market_features = market_features.loc[common_index].sort_index()
macro_features = macro_features.loc[common_index].sort_index()
asset_targets = asset_targets.loc[common_index].sort_index()

print(f"\nAligned data shape: {market_features.shape}")
print(f"Date range: {market_features.index[0]} to {market_features.index[-1]}")

# Combine features for regime detection
all_features = pd.concat([market_features, macro_features], axis=1)
print(f"Combined features for HMM: {all_features.shape}")
print(f"Features: {list(all_features.columns)[:10]}...

SyntaxError: unterminated f-string literal (detected at line 27) (1248322929.py, line 27)

## 3. Gaussian HMM Regime Detection

In [ ]:
# Split into Train/Validation/Test (2014-2020 / 2021-2022 / 2023-2026)
split_date_train = '2020-12-31'
split_date_val = '2022-12-31'

train_mask = all_features.index <= split_date_train
val_mask = (all_features.index > split_date_train) & (all_features.index <= split_date_val)
test_mask = all_features.index > split_date_val

train_features = all_features[train_mask].reset_index(drop=True)
train_returns = asset_targets[train_mask]

val_features = all_features[val_mask].reset_index(drop=True)
val_returns = asset_targets[val_mask]

test_features = all_features[test_mask].reset_index(drop=True)
test_returns = asset_targets[test_mask]

# Select only numeric columns and drop NaN
numeric_cols = train_features.select_dtypes(include=[np.number]).columns
train_features = train_features[numeric_cols].fillna(0)
val_features = val_features[numeric_cols].fillna(0)
test_features = test_features[numeric_cols].fillna(0)

print(f"Train data: {len(train_features)} weeks")
print(f"Val data: {len(val_features)} weeks")
print(f"Test data: {len(test_features)} weeks")
print(f"Train features shape: {train_features.shape} (numeric features only)")

# Fit HMM on training data
print("\n=== Fit Gaussian HMM ===")
hmm = GaussianHMMRegimeDetector(n_regimes=4, pca_components=10, random_state=42)
hmm.fit(train_features, regime_names=["Risk-On", "Neutral", "Defensive", "Panic"])

print(f"HMM fitted with {hmm.n_regimes} regimes")
print(f"Regime names: {hmm.get_regime_names()}")

# Predict regimes for all periods
train_regime_posteriors = hmm.predict_proba(train_features)
val_regime_posteriors = hmm.predict_proba(val_features)
test_regime_posteriors = hmm.predict_proba(test_features)

print(f"\nRegime posteriors shape:")
print(f"  Train: {train_regime_posteriors.shape}")
print(f"  Val: {val_regime_posteriors.shape}")
print(f"  Test: {test_regime_posteriors.shape}")

# Visualize regime detection with Plotly
import plotly.graph_objects as go
from plotly.subplots import make_subplots

predicted_regimes = hmm.predict_regimes(train_features)
fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=False,
    vertical_spacing=0.12,
    subplot_titles=(
        'Market Regimes (Train Period) - HMM Posterior Probabilities',
        'Most Likely Market Regime Over Time',
    ),
)

for regime_id in range(hmm.n_regimes):
    fig.add_trace(
        go.Scatter(
            x=list(range(len(train_regime_posteriors))),
            y=train_regime_posteriors[:, regime_id],
            mode='lines',
            name=hmm.regime_names[regime_id],
        ),
        row=1,
        col=1,
    )

fig.add_trace(
    go.Scatter(
        x=list(range(len(predicted_regimes))),
        y=predicted_regimes,
        mode='markers',
        marker=dict(size=6, opacity=0.7),
        name='Predicted Regime',
        showlegend=False,
    ),
    row=2,
    col=1,
)

fig.update_xaxes(title_text='Week', row=1, col=1)
fig.update_yaxes(title_text='Probability', row=1, col=1)
fig.update_xaxes(title_text='Week', row=2, col=1)
fig.update_yaxes(
    title_text='Regime ID',
    tickmode='array',
    tickvals=list(range(hmm.n_regimes)),
    ticktext=hmm.regime_names,
    row=2,
    col=1,
)
fig.update_layout(height=820, width=1250, legend=dict(orientation='h', y=1.02, x=0))
fig.show()

print("\nRegime Statistics (Train Period):")
regime_stats = hmm.get_regime_stats()
for regime_id, stats in regime_stats.items():
    print(f"\n{hmm.regime_names[regime_id]}:")
    for feature, value in list(stats.items())[:5]:
        print(f"  {feature}: {value:.4f}")

## 4. Create Training, Validation, and Test Environments

In [ ]:
# Asset returns for portfolio: [SPY, TLT, GLD, Cash(~0)]
# Check available columns and use those
print(f"Available columns in asset_targets: {asset_targets.columns.tolist()}")

# Use columns that match the expected assets
spy_col = 'SPY_return' if 'SPY_return' in asset_targets.columns else next((c for c in asset_targets.columns if 'SPY' in c.upper()), None)
tlt_col = 'TLT_return' if 'TLT_return' in asset_targets.columns else next((c for c in asset_targets.columns if 'TLT' in c.upper()), None)
gld_col = 'GLD_return' if 'GLD_return' in asset_targets.columns else next((c for c in asset_targets.columns if 'GLD' in c.upper()), None)

print(f"Using columns: SPY={spy_col}, TLT={tlt_col}, GLD={gld_col}")

spy_returns = asset_targets[spy_col] if spy_col else np.random.randn(len(asset_targets)) * 0.02
tlt_returns = asset_targets[tlt_col] if tlt_col else np.random.randn(len(asset_targets)) * 0.01
gld_returns = asset_targets[gld_col] if gld_col else np.random.randn(len(asset_targets)) * 0.015
cash_returns = np.zeros_like(spy_returns)  # Cash has ~0 return

portfolio_returns = pd.DataFrame({
    'SPY': spy_returns.values if hasattr(spy_returns, 'values') else spy_returns,
    'TLT': tlt_returns.values if hasattr(tlt_returns, 'values') else tlt_returns,
    'GLD': gld_returns.values if hasattr(gld_returns, 'values') else gld_returns,
    'Cash': cash_returns,
})

# Normalize features (important for neural network)
scaler = StandardScaler()
train_market_scaled = scaler.fit_transform(train_features)
val_market_scaled = scaler.transform(val_features)
test_market_scaled = scaler.transform(test_features)

train_features_normalized = train_features.copy()
train_features_normalized[:] = train_market_scaled

val_features_normalized = val_features.copy()
val_features_normalized[:] = val_market_scaled

test_features_normalized = test_features.copy()
test_features_normalized[:] = test_market_scaled

# Create environments from centralized config
train_env = WeeklyPortfolioEnv(
    features=train_features_normalized,
    regime_posteriors=train_regime_posteriors,
    asset_returns=portfolio_returns.iloc[:len(train_features_normalized)],
    transaction_cost=env_cfg['transaction_cost'],
    volatility_penalty=env_cfg['volatility_penalty'],
    lookback_vol=env_cfg['lookback_vol'],
    seq_len=SEQ_LEN
)

val_env = WeeklyPortfolioEnv(
    features=val_features_normalized,
    regime_posteriors=val_regime_posteriors,
    asset_returns=portfolio_returns.iloc[len(train_features_normalized):len(train_features_normalized)+len(val_features_normalized)],
    transaction_cost=env_cfg['transaction_cost'],
    volatility_penalty=env_cfg['volatility_penalty'],
    lookback_vol=env_cfg['lookback_vol'],
    seq_len=SEQ_LEN
)

test_env = WeeklyPortfolioEnv(
    features=test_features_normalized,
    regime_posteriors=test_regime_posteriors,
    asset_returns=portfolio_returns.iloc[len(train_features_normalized)+len(val_features_normalized):],
    transaction_cost=env_cfg['transaction_cost'],
    volatility_penalty=env_cfg['volatility_penalty'],
    lookback_vol=env_cfg['lookback_vol'],
    seq_len=SEQ_LEN
)

print(f"Environment Configuration:")
print(f"  State dimension: {train_env.observation_space.shape}")
print(f"  Action dimension: {train_env.action_space.n}")
print(f"  Sequence length: {train_env.seq_len}")
print(f"  Portfolio templates: {len(train_env.PORTFOLIO_TEMPLATES)}")
print(f"\nPortfolio Actions:")
for action_id, name in enumerate(train_env.ACTION_NAMES):
    print(f"  {action_id}: {name}")

## 5. Initialize Attention DQN Agent

In [ ]:
# Initialize DQN agent using centralized hyperparameter config
agent = DQN(
    'MlpPolicy',
    train_env,
    learning_rate=dqn_cfg['learning_rate'],
    buffer_size=dqn_cfg['buffer_size'],
    learning_starts=dqn_cfg['learning_starts'],
    batch_size=dqn_cfg['batch_size'],
    tau=1.0,
    gamma=dqn_cfg['gamma'],
    train_freq=1,
    target_update_interval=dqn_cfg['target_update_interval'],
    exploration_fraction=dqn_cfg['exploration_fraction'],
    exploration_initial_eps=dqn_cfg['exploration_initial_eps'],
    exploration_final_eps=dqn_cfg['exploration_final_eps'],
    device=device,
    verbose=1
)

print("=== FinRL DQN Agent Initialized ===")
print(f"Device: {agent.device}")
print(f"Policy: MlpPolicy")
print(f"State space: {train_env.observation_space}")
print(f"Action space: {train_env.action_space}")
print(f"Learning rate: {agent.learning_rate}")
print(f"Gamma: {agent.gamma}")
print(f"\nBuffer size: {agent.buffer_size}")
print(f"Batch size: {agent.batch_size}")
print(f"Target update interval: {agent.target_update_interval}")

# Test forward pass (gymnasium returns (obs, info) tuple)
obs, info = train_env.reset()
print(f"\nObservation shape: {obs.shape}")
print(f"Info keys: {info.keys() if isinstance(info, dict) else 'N/A'}")

## 6. Train Attention-Enhanced DQN Agent

In [ ]:
# Load saved DQN weights for explainability and diagnostics
from stable_baselines3 import DQN

dqn_model_path = MODEL_PATHS['dqn']
if not dqn_model_path.exists():
    raise FileNotFoundError(
        f"Missing DQN weights at {dqn_model_path}. Run notebooks_split/01_training_and_tuning.ipynb first."
    )

agent = DQN.load(str(dqn_model_path), env=test_env, device=device)
print(f"Loaded DQN weights from: {dqn_model_path}")

test_eval = evaluate_episode(agent, test_env, deterministic=True)
actions_df = pd.DataFrame(test_eval.get('actions', []))
val_history = []

In [ ]:
# Plot training progress with validation rewards (Plotly)
import plotly.graph_objects as go
from plotly.subplots import make_subplots

if 'val_history' not in globals() or not val_history:
    print('No validation history found. Run training/evaluation cells that create val_history first.')
else:
    val_steps = [h['step'] for h in val_history]
    val_rewards = [h['reward'] for h in val_history]
    reward_improvement = [val_rewards[0]] + val_rewards
    best_idx = int(np.argmax(val_rewards))

    fig = make_subplots(
        rows=2,
        cols=2,
        subplot_titles=(
            'Validation Rewards During Training',
            'Reward Progression',
            'Validation Rewards (Best Performance)',
            'Training Summary',
        ),
        specs=[[{'type': 'xy'}, {'type': 'xy'}], [{'type': 'xy'}, {'type': 'table'}]],
        vertical_spacing=0.12,
    )

    fig.add_trace(
        go.Scatter(x=val_steps, y=val_rewards, mode='lines+markers', name='Validation Reward'),
        row=1,
        col=1,
    )

    fig.add_trace(
        go.Scatter(x=list(range(len(reward_improvement))), y=reward_improvement, mode='lines+markers',
                   name='Reward Progression', line=dict(color='orange')),
        row=1,
        col=2,
    )

    fig.add_trace(
        go.Scatter(x=list(range(len(val_rewards))), y=val_rewards, mode='lines+markers',
                   name='Val Rewards (Best)', showlegend=False),
        row=2,
        col=1,
    )
    fig.add_hline(y=val_rewards[best_idx], line_dash='dash', line_color='green', row=2, col=1)

    summary_rows = [
        ['Total timesteps', f"{training_results.get('total_timesteps', 0):,}"],
        ['Validations run', f"{len(val_history)}"],
        ['Best val reward', f"{training_results.get('best_val_reward', max(val_rewards)):.4f}"],
        ['Final val reward', f"{val_rewards[-1]:.4f}"],
        ['Learning rate', str(dqn_cfg.get('learning_rate'))],
        ['Gamma', str(dqn_cfg.get('gamma'))],
        ['Buffer size', str(dqn_cfg.get('buffer_size'))],
        ['Batch size', str(dqn_cfg.get('batch_size'))],
    ]

    fig.add_trace(
        go.Table(
            header=dict(values=['Metric', 'Value']),
            cells=dict(values=[[r[0] for r in summary_rows], [r[1] for r in summary_rows]]),
        ),
        row=2,
        col=2,
    )

    fig.update_xaxes(title_text='Timesteps', row=1, col=1)
    fig.update_yaxes(title_text='Reward', row=1, col=1)
    fig.update_xaxes(title_text='Validation Checkpoint', row=1, col=2)
    fig.update_yaxes(title_text='Reward', row=1, col=2)
    fig.update_xaxes(title_text='Validation Checkpoint', row=2, col=1)
    fig.update_yaxes(title_text='Reward', row=2, col=1)
    fig.update_layout(height=900, width=1400, legend=dict(orientation='h', y=1.02, x=0))
    fig.show()

    print('Training Statistics:')
    print(f"  Validation checkpoints: {len(val_history)}")
    print(f"  Best validation reward: {max(val_rewards):.4f}")
    print(f"  Final validation reward: {val_rewards[-1]:.4f}")
    print(f"  Reward improvement: {val_rewards[-1] - val_rewards[0]:.4f}")

## 8. Evaluate Agent on Test Period

In [ ]:
# Evaluate FinRL agent on test set
print("=== Evaluating FinRL DQN on Test Period ===\n")

import plotly.graph_objects as go
from plotly.subplots import make_subplots

test_eval = evaluate_episode(agent, test_env, deterministic=True)

print("Test Period Performance:")
print(f"  Total Reward: {test_eval['reward']:.4f}")
print(f"  Cumulative Return: {test_eval.get('cumulative_return', 0):.4f}")
print(f"  Sharpe Ratio: {test_eval.get('sharpe_ratio', 0):.4f}")
print(f"  Max Drawdown: {test_eval.get('max_drawdown', 0):.4f}")
print(f"  Episode Length: {test_eval['length']}")

# Extract action sequence
actions_df = pd.DataFrame(test_eval['actions'])
print(f"\nAction Distribution:")
print(actions_df['action_name'].value_counts())

# Weekly performance (Plotly)
cumulative_test_returns = np.cumprod(1 + actions_df['return']) - 1

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.10,
                    subplot_titles=('Weekly Portfolio Returns (Test Period)', 'Cumulative Portfolio Returns (Test Period)'))

fig.add_trace(
    go.Scatter(x=actions_df.index, y=actions_df['return'], mode='lines+markers', name='Weekly Return'),
    row=1,
    col=1,
)
fig.add_hline(y=0, line_dash='dash', line_color='red', opacity=0.4, row=1, col=1)

fig.add_trace(
    go.Scatter(x=actions_df.index, y=cumulative_test_returns, mode='lines+markers',
               fill='tozeroy', name='Cumulative Return', line=dict(color='green')),
    row=2,
    col=1,
)

fig.update_xaxes(title_text='Week', row=2, col=1)
fig.update_yaxes(title_text='Return', row=1, col=1)
fig.update_yaxes(title_text='Cumulative Return', row=2, col=1)
fig.update_layout(height=760, width=1300, legend=dict(orientation='h', y=1.02, x=0))
fig.show()

## 9. Analyze Attention Weights and Saliency

In [ ]:
# Plot BOTH: (1) attention weights (from an attention-based surrogate model)
# and (2) saliency (from gradients of the actual FinRL DQN), using Plotly.
print("=== Attention Weights + Saliency ===\n")

import torch
import torch.nn as nn
import torch.optim as optim
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# -----------------------------
# 1) Build a dataset of states
# -----------------------------
seq_len = train_env.seq_len
feat_dim = train_env.observation_space.shape[-1]
n_collect = 240

obs, _ = test_env.reset()
state_list = []

for _ in range(n_collect):
    state_list.append(obs.copy())
    action, _ = agent.predict(obs, deterministic=True)
    action_val = int(action.item() if isinstance(action, np.ndarray) else action)
    obs, reward, terminated, truncated, _ = test_env.step(action_val)
    if terminated or truncated:
        obs, _ = test_env.reset()

X_np = np.array(state_list, dtype=np.float32)  # [N, T, F]
X = torch.tensor(X_np, dtype=torch.float32, device=agent.device)

with torch.no_grad():
    y_q = agent.q_net(X).detach()  # target Q-values from trained DQN

# -----------------------------------------------
# 2) Attention surrogate to mimic DQN Q-function
# -----------------------------------------------
class AttentionQSurrogate(nn.Module):
    def __init__(self, in_dim, d_model=64, n_heads=4, n_actions=7):
        super().__init__()
        self.proj = nn.Linear(in_dim, d_model)
        self.attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.query = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, 64),
            nn.ReLU(),
            nn.Linear(64, n_actions)
        )

    def forward(self, x):
        h = torch.relu(self.proj(x))
        h_attn, attn_w = self.attn(h, h, h, need_weights=True, average_attn_weights=False)
        q = self.query.expand(h_attn.size(0), -1, -1)
        pooled, pool_w = self.attn(q, h_attn, h_attn, need_weights=True, average_attn_weights=False)
        q_pred = self.mlp(pooled.squeeze(1))
        return q_pred, attn_w, pool_w

n_actions = train_env.action_space.n
surrogate = AttentionQSurrogate(feat_dim, d_model=64, n_heads=4, n_actions=n_actions).to(agent.device)
criterion = nn.MSELoss()
optimizer = optim.Adam(surrogate.parameters(), lr=1e-3)

batch_size = 64
epochs = 30
num_samples = X.size(0)

for epoch in range(epochs):
    perm = torch.randperm(num_samples, device=agent.device)
    epoch_loss = 0.0

    for i in range(0, num_samples, batch_size):
        idx = perm[i:i + batch_size]
        xb = X[idx]
        yb = y_q[idx]

        pred, _, _ = surrogate(xb)
        loss = criterion(pred, yb)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item() * xb.size(0)

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{epochs} - surrogate MSE: {epoch_loss/num_samples:.6f}")

# -----------------------------------------------------
# 3) Extract ATTENTION WEIGHTS from the surrogate model
# -----------------------------------------------------
with torch.no_grad():
    q_pred_all, attn_w_all, pool_w_all = surrogate(X)

attn_matrix_mean = attn_w_all.mean(dim=(0, 1)).detach().cpu().numpy()  # [T, T]
temporal_attn_mean = pool_w_all.mean(dim=(0, 1, 2)).detach().cpu().numpy()  # [T]
temporal_attn_mean = temporal_attn_mean / (temporal_attn_mean.sum() + 1e-12)

# ---------------------------------------
# 4) Compute SALIENCY from actual DQN
# ---------------------------------------
n_sal = min(120, X_np.shape[0])
saliency_timestep = []

for i in range(n_sal):
    obs_leaf = torch.tensor(X_np[i], dtype=torch.float32, device=agent.device, requires_grad=True)
    q_vals = agent.q_net(obs_leaf.unsqueeze(0))
    best_a = int(torch.argmax(q_vals, dim=1).item())
    score = q_vals[0, best_a]

    agent.q_net.zero_grad(set_to_none=True)
    score.backward()

    grad_abs = obs_leaf.grad.detach().abs().cpu().numpy()  # [T, F]
    sal_t = grad_abs.sum(axis=1)
    sal_t = sal_t / (sal_t.sum() + 1e-12)
    saliency_timestep.append(sal_t)

saliency_timestep = np.array(saliency_timestep)
saliency_timestep_mean = saliency_timestep.mean(axis=0)

token_labels = [f"t-{seq_len-1-i}" for i in range(seq_len)]

# ---------------------------------------
# 5) Plotly dashboard: attention + saliency
# ---------------------------------------
n_heat = min(40, saliency_timestep.shape[0])
x = list(range(seq_len))

fig = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=(
        'Surrogate Self-Attention Matrix (True Attention Weights)',
        'Temporal Importance: Attention vs Saliency',
        'Saliency Heatmap (Actual DQN)',
        'Per-Timestep Comparison',
    ),
    specs=[[{'type': 'heatmap'}, {'type': 'xy'}], [{'type': 'heatmap'}, {'type': 'xy'}]],
)

fig.add_trace(
    go.Heatmap(z=attn_matrix_mean, x=token_labels, y=token_labels, colorscale='Blues', colorbar=dict(title='weight')),
    row=1,
    col=1,
)

fig.add_trace(
    go.Scatter(x=token_labels, y=temporal_attn_mean, mode='lines+markers', name='Attention weights', line=dict(color='steelblue', width=3)),
    row=1,
    col=2,
)
fig.add_trace(
    go.Scatter(x=token_labels, y=saliency_timestep_mean, mode='lines+markers', name='Saliency', line=dict(color='darkorange', width=3, dash='dot')),
    row=1,
    col=2,
)

fig.add_trace(
    go.Heatmap(z=saliency_timestep[:n_heat], x=token_labels, y=list(range(n_heat)), colorscale='YlOrRd', colorbar=dict(title='saliency')),
    row=2,
    col=1,
)

fig.add_trace(
    go.Bar(x=token_labels, y=temporal_attn_mean, name='Attention weights', marker_color='steelblue', opacity=0.85),
    row=2,
    col=2,
)
fig.add_trace(
    go.Bar(x=token_labels, y=saliency_timestep_mean, name='Saliency', marker_color='darkorange', opacity=0.75),
    row=2,
    col=2,
)

fig.update_layout(height=980, width=1450, barmode='group', legend=dict(orientation='h', y=1.02, x=0))
fig.update_xaxes(title_text='Key timestep', row=1, col=1)
fig.update_yaxes(title_text='Query timestep', row=1, col=1)
fig.update_xaxes(title_text='Timestep in state window', row=1, col=2)
fig.update_yaxes(title_text='Normalized importance', row=1, col=2)
fig.update_xaxes(title_text='Timestep in state window', row=2, col=1)
fig.update_yaxes(title_text='Sample index', row=2, col=1)
fig.update_xaxes(title_text='Timestep in state window', row=2, col=2)
fig.update_yaxes(title_text='Normalized importance', row=2, col=2)
fig.show()

print("Interpretation:")
print("- Top-left is TRUE attention weights from the surrogate attention model.")
print("- Bottom-left and orange curves are gradient saliency from the actual DQN.")
print("- If both agree on timesteps, attribution is more robust.")
print("- If they diverge, policy sensitivity and learned temporal weighting differ.")

In [ ]:
# Visual explanation panel (Plotly split view)
print("=== Visual Attention Storyboard (Split View) ===")

import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Ensure arrays are on CPU/NumPy
attn_global = np.array(temporal_attn_mean, dtype=float)
sal_global = np.array(saliency_timestep_mean, dtype=float)

attn_global = attn_global / (attn_global.sum() + 1e-12)
sal_global = sal_global / (sal_global.sum() + 1e-12)

# Build readable token labels (oldest -> newest)
tokens = [f"t-{seq_len-1-i}" for i in range(seq_len)]

# ------------------------------------------------------------------
# Figure 1: Token flow diagram (Sankey)
# ------------------------------------------------------------------
node_labels = tokens + ['Attention Query', 'Chosen Action']
query_idx = len(tokens)
action_idx = len(tokens) + 1

sources, targets, values, colors = [], [], [], []
for i, w in enumerate(attn_global):
    sources.append(i)
    targets.append(query_idx)
    values.append(float(w + 1e-6))
    colors.append('rgba(44,127,184,0.45)')
for i, w in enumerate(sal_global):
    sources.append(i)
    targets.append(action_idx)
    values.append(float(w + 1e-6))
    colors.append('rgba(230,126,34,0.45)')
sources.append(query_idx)
targets.append(action_idx)
values.append(float(max(attn_global.sum(), 1e-6)))
colors.append('rgba(168,78,28,0.70)')

fig_flow = go.Figure(
    data=[
        go.Sankey(
            arrangement='snap',
            node=dict(label=node_labels, pad=15, thickness=14),
            link=dict(source=sources, target=targets, value=values, color=colors),
        )
    ]
)
fig_flow.update_layout(title='A) Temporal Token Flow: Attention Path vs Saliency Path', height=620, width=1300)
fig_flow.show()

# ------------------------------------------------------------------
# Figure 2: Attention examples only
# ------------------------------------------------------------------
n_examples = min(4, saliency_timestep.shape[0])
attn_per_sample = pool_w_all.mean(dim=1).squeeze(1).detach().cpu().numpy()[:n_examples]

fig_attn = make_subplots(rows=1, cols=n_examples, subplot_titles=[f"({['i','ii','iii','iv'][j]}) Attention" for j in range(n_examples)])
for j in range(n_examples):
    vals = attn_per_sample[j]
    fig_attn.add_trace(go.Bar(x=tokens, y=vals, showlegend=False, marker_color='#5dade2'), row=1, col=j+1)
    q_idx = int(np.argmax(vals))
    fig_attn.add_trace(go.Scatter(x=[tokens[q_idx]], y=[vals[q_idx]], mode='markers', showlegend=False,
                                  marker=dict(size=14, color='#d81b60', symbol='star')), row=1, col=j+1)
fig_attn.update_layout(height=420, width=max(900, 320*n_examples), title='B) Query Examples - Attention Weights')
fig_attn.show()

# ------------------------------------------------------------------
# Figure 3: Saliency examples only
# ------------------------------------------------------------------
sal_per_sample = saliency_timestep[:n_examples]
fig_sal = make_subplots(rows=1, cols=n_examples, subplot_titles=[f"({['i','ii','iii','iv'][j]}) Saliency" for j in range(n_examples)])
for j in range(n_examples):
    hm = sal_per_sample[j].reshape(1, -1)
    fig_sal.add_trace(
        go.Heatmap(z=hm, x=tokens, y=[''], colorscale='Magma', showscale=(j == n_examples - 1), colorbar=dict(title='saliency')),
        row=1,
        col=j+1,
    )
fig_sal.update_layout(height=360, width=max(900, 320*n_examples), title='C) Query Examples - Saliency Maps')
fig_sal.show()

print("How to read this split view:")
print("1) Figure A shows information pathways (attention path vs saliency path).")
print("2) Figure B shows which token each query attends to most (star marker).")
print("3) Figure C shows where the actual DQN is most sensitive (brighter = stronger saliency).")

## 9.1 Interactive Plotly View (Token Flow + Attention vs Saliency)

In [ ]:
# Interactive Plotly view: token flow + temporal comparison
print("=== Interactive Plotly Attention View ===")

import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Safety checks in case prior cells were not run
if 'temporal_attn_mean' not in globals() or 'saliency_timestep_mean' not in globals():
    raise RuntimeError("Please run Cell 19 first to generate attention/saliency arrays.")

seq_len_local = int(seq_len) if 'seq_len' in globals() else len(temporal_attn_mean)
tokens_local = [f"t-{seq_len_local-1-i}" for i in range(seq_len_local)]

attn_global = np.array(temporal_attn_mean, dtype=float)
sal_global = np.array(saliency_timestep_mean, dtype=float)
attn_global = attn_global / (attn_global.sum() + 1e-12)
sal_global = sal_global / (sal_global.sum() + 1e-12)

# --- Figure 1: Sankey token flow ---
node_labels = [f"Memory {t}" for t in tokens_local] + ["Attention Query", "Chosen Action"]
query_idx = len(tokens_local)
action_idx = len(tokens_local) + 1

sources = []
targets = []
values = []
link_colors = []

# Memory -> Query (attention)
for i in range(len(tokens_local)):
    sources.append(i)
    targets.append(query_idx)
    values.append(float(attn_global[i]))
    link_colors.append("rgba(44,127,184,0.70)")

# Memory -> Action (saliency)
for i in range(len(tokens_local)):
    sources.append(i)
    targets.append(action_idx)
    values.append(float(sal_global[i]))
    link_colors.append("rgba(230,126,34,0.55)")

# Query -> Action
sources.append(query_idx)
targets.append(action_idx)
values.append(1.0)
link_colors.append("rgba(168,78,28,0.85)")

fig_flow = go.Figure(
    data=[go.Sankey(
        arrangement="snap",
        node=dict(
            pad=20,
            thickness=22,
            line=dict(color="black", width=0.4),
            label=node_labels,
            color=["#d8ead3"] * len(tokens_local) + ["#e6d9f2", "#fde2cf"],
        ),
        link=dict(
            source=sources,
            target=targets,
            value=values,
            color=link_colors,
            hovertemplate="%{source.label} -> %{target.label}<br>weight=%{value:.3f}<extra></extra>",
        ),
    )]
)

fig_flow.update_layout(
    title="Interactive Token Flow: Attention (blue) vs Saliency (orange)",
    font=dict(size=12),
    height=520,
    width=1400
)
fig_flow.show()

# --- Figure 2: Temporal comparison + sample heatmap ---
n_examples_local = min(4, saliency_timestep.shape[0])
attn_per_sample_local = pool_w_all.mean(dim=1).squeeze(1).detach().cpu().numpy()[:n_examples_local]
sal_per_sample_local = saliency_timestep[:n_examples_local]

fig_compare = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=(
        "Temporal Importance (Global)",
        "Delta from Uniform Baseline",
        "Attention per Query Example",
        "Saliency per Query Example",
    ),
    specs=[[{"type": "xy"}, {"type": "xy"}], [{"type": "heatmap"}, {"type": "heatmap"}]],
    horizontal_spacing=0.12,
    vertical_spacing=0.18,
)

uniform = np.ones_like(attn_global) / len(attn_global)

fig_compare.add_trace(
    go.Scatter(x=tokens_local, y=attn_global, mode="lines+markers", name="Attention", line=dict(color="#2c7fb8", width=3)),
    row=1, col=1
)
fig_compare.add_trace(
    go.Scatter(x=tokens_local, y=sal_global, mode="lines+markers", name="Saliency", line=dict(color="#e67e22", width=3)),
    row=1, col=1
)

fig_compare.add_trace(
    go.Bar(x=tokens_local, y=attn_global - uniform, name="Attention - Uniform", marker_color="#5dade2"),
    row=1, col=2
)
fig_compare.add_trace(
    go.Bar(x=tokens_local, y=sal_global - uniform, name="Saliency - Uniform", marker_color="#f5b041"),
    row=1, col=2
)

fig_compare.add_trace(
    go.Heatmap(
        z=attn_per_sample_local,
        x=tokens_local,
        y=[f"query_{i+1}" for i in range(n_examples_local)],
        colorscale="Blues",
        colorbar=dict(title="attn", x=0.46),
        showscale=True,
        hovertemplate="query=%{y}<br>token=%{x}<br>attn=%{z:.3f}<extra></extra>",
        name="Attention heatmap",
    ),
    row=2, col=1
)

fig_compare.add_trace(
    go.Heatmap(
        z=sal_per_sample_local,
        x=tokens_local,
        y=[f"query_{i+1}" for i in range(n_examples_local)],
        colorscale="Oranges",
        colorbar=dict(title="sal", x=1.02),
        showscale=True,
        hovertemplate="query=%{y}<br>token=%{x}<br>saliency=%{z:.3f}<extra></extra>",
        name="Saliency heatmap",
    ),
    row=2, col=2
)

fig_compare.update_layout(
    height=760,
    width=1400,
    title="Interactive Attention vs Saliency Diagnostics",
    barmode="group",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
)
fig_compare.update_yaxes(title_text="importance", row=1, col=1)
fig_compare.update_yaxes(title_text="delta", row=1, col=2)
fig_compare.show()

print("Why Plotly can be better here:")
print("1) Hover gives exact weights per token instead of eyeballing bar/heatmap colors.")
print("2) Sankey makes the query->action relation visually concrete.")
print("3) Zoom/pan helps compare tiny differences when attention looks nearly uniform.")

In [ ]:
# Inference utilities and rolling inference demo
print("=== Inference Demo ===")

import plotly.express as px


def infer_action(agent, obs, action_names=None, deterministic=True):
    """Run one-step inference from a single observation window."""
    action, _ = agent.predict(obs, deterministic=deterministic)
    action_id = int(action.item() if isinstance(action, np.ndarray) else action)

    # Also expose Q-values for transparency
    with torch.no_grad():
        q_vals = agent.q_net(torch.tensor(obs, dtype=torch.float32, device=agent.device).unsqueeze(0))
        q_vals = q_vals.detach().cpu().numpy()[0]

    return {
        'action_id': action_id,
        'action_name': action_names[action_id] if action_names is not None else f'Action {action_id}',
        'q_values': q_vals,
        'confidence_gap': float(np.max(q_vals) - np.partition(q_vals, -2)[-2]) if len(q_vals) > 1 else 0.0,
    }


# 1) Single-state inference example
obs0, _ = test_env.reset()
infer0 = infer_action(agent, obs0, action_names=train_env.ACTION_NAMES)

print("Single-step inference:")
print(f"  Predicted action: {infer0['action_id']} ({infer0['action_name']})")
print(f"  Confidence gap (top1-top2 Q): {infer0['confidence_gap']:.6f}")

# Q-values for the single state
q_df = pd.DataFrame({
    'action_id': np.arange(len(infer0['q_values'])),
    'action_name': train_env.ACTION_NAMES,
    'q_value': infer0['q_values']
}).sort_values('q_value', ascending=False)

fig_q = px.bar(
    q_df,
    x='action_name',
    y='q_value',
    title='Inference Q-Values for One State',
    labels={'action_name': 'Action', 'q_value': 'Q-value'}
)
fig_q.update_layout(xaxis_tickangle=-25)
fig_q.show()

# 2) Rolling inference across test period
obs, _ = test_env.reset()
rows = []
max_steps = min(160, len(test_env.features))

for t in range(max_steps):
    inf = infer_action(agent, obs, action_names=train_env.ACTION_NAMES)

    rows.append({
        'step': t,
        'action_id': inf['action_id'],
        'action_name': inf['action_name'],
        'confidence_gap': inf['confidence_gap'],
        'q_max': float(np.max(inf['q_values'])),
    })

    obs, reward, terminated, truncated, _ = test_env.step(inf['action_id'])
    if terminated or truncated:
        break

infer_df = pd.DataFrame(rows)

print(f"\nRolling inference steps: {len(infer_df)}")
print("Top predicted actions:")
print(infer_df['action_name'].value_counts().head(5))

# Interactive timeline of chosen actions
fig_actions = px.scatter(
    infer_df,
    x='step',
    y='action_name',
    color='confidence_gap',
    color_continuous_scale='Viridis',
    title='Rolling Inference: Predicted Actions over Time',
    labels={'step': 'Inference step', 'action_name': 'Predicted action', 'confidence_gap': 'Confidence gap'}
)
fig_actions.show()

# Confidence trend
fig_conf = px.line(
    infer_df,
    x='step',
    y='confidence_gap',
    title='Model Confidence Gap over Time (Top1 - Top2 Q)',
    labels={'step': 'Inference step', 'confidence_gap': 'Confidence gap'}
)
fig_conf.show()

## 9.2 Finance Explainability Dashboard (DQN)

This section links DQN attention/saliency outputs to finance context (returns, actions, and inferred regimes).
Run Section 8 and Section 9 first before executing the dashboard cell.

In [ ]:
# 9.2 Finance-native Plotly explainability dashboard (attention + action + regime)
print("=== Explainable RL (Finance View): Attention vs Returns vs Regime ===")

import importlib
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ml.explainability_plotly as xai_plotly
xai_plotly = importlib.reload(xai_plotly)

required_vars = [
    "temporal_attn_mean",
    "saliency_timestep_mean",
    "pool_w_all",
    "saliency_timestep",
    "actions_df",
]
missing = [name for name in required_vars if name not in globals()]
if missing:
    raise RuntimeError(
        "Missing variables for finance dashboard: "
        + ", ".join(missing)
        + ". Run Section 8 and Section 9 first."
    )

# --- Build aligned finance explainability table ---
tokens_local = [f"t-{len(temporal_attn_mean)-1-i}" for i in range(len(temporal_attn_mean))]
attn_per_sample = pool_w_all.mean(dim=1).squeeze(1).detach().cpu().numpy()
sal_per_sample = np.array(saliency_timestep, dtype=float)

n_common = min(len(actions_df), len(attn_per_sample), len(sal_per_sample))
if n_common < 10:
    raise RuntimeError("Not enough aligned samples for finance dashboard. Re-run explainability extraction.")

act_local = actions_df.iloc[:n_common].reset_index(drop=True).copy()
attn_local = attn_per_sample[:n_common]
sal_local = sal_per_sample[:n_common]

act_local["weekly_return"] = act_local["return"].astype(float)
act_local["cum_return"] = (1.0 + act_local["weekly_return"]).cumprod() - 1.0
act_local["attn_recent"] = attn_local[:, -1]
act_local["sal_recent"] = sal_local[:, -1]
act_local["attn_entropy"] = -np.sum(attn_local * np.log(attn_local + 1e-12), axis=1)
act_local["dominant_lag"] = np.argmax(attn_local, axis=1)
act_local["dominant_token"] = [tokens_local[i] for i in act_local["dominant_lag"]]

# Optional regime alignment (from HMM predictions)
if "predicted_regimes" in globals():
    reg = np.asarray(predicted_regimes)
    tail = reg[-n_common:] if len(reg) >= n_common else np.pad(reg, (n_common - len(reg), 0), mode="edge")
    act_local["regime_id"] = tail.astype(int)
    if "hmm" in globals() and hasattr(hmm, "regime_names"):
        act_local["regime_name"] = [hmm.regime_names[i] for i in act_local["regime_id"]]
    else:
        act_local["regime_name"] = [f"Regime_{i}" for i in act_local["regime_id"]]
else:
    act_local["regime_name"] = "Unknown"

# --- Figure 1: Portfolio performance + temporal attention map ---
show_n = min(100, n_common)
view_df = act_local.iloc[:show_n].copy()

fig_fin = make_subplots(
    rows=3,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.08,
    row_heights=[0.34, 0.40, 0.26],
    specs=[[{"type": "xy"}], [{"type": "heatmap"}], [{"type": "xy"}]],
    subplot_titles=(
        "Portfolio Path with Attention-to-Recent Signal",
        "Temporal Attention Heatmap (older to newer state tokens)",
        "Action Timeline + Regime Context",
    ),
)

fig_fin.add_trace(
    go.Scatter(
        x=view_df.index,
        y=view_df["cum_return"],
        mode="lines+markers",
        name="Cumulative return",
        line=dict(color="#115e59", width=3),
        marker=dict(size=5),
        hovertemplate="step=%{x}<br>cum_return=%{y:.2%}<extra></extra>",
    ),
    row=1,
    col=1,
)

fig_fin.add_trace(
    go.Bar(
        x=view_df.index,
        y=view_df["weekly_return"],
        name="Weekly return",
        marker_color="#94a3b8",
        opacity=0.45,
        hovertemplate="step=%{x}<br>weekly_return=%{y:.2%}<extra></extra>",
    ),
    row=1,
    col=1,
)

fig_fin.add_trace(
    go.Scatter(
        x=view_df.index,
        y=view_df["attn_recent"],
        mode="lines",
        name="Attention on most recent token",
        line=dict(color="#b45309", width=2, dash="dot"),
        hovertemplate="step=%{x}<br>attn_recent=%{y:.3f}<extra></extra>",
    ),
    row=1,
    col=1,
)

fig_fin.add_trace(
    go.Heatmap(
        z=attn_local[:show_n].T,
        x=list(range(show_n)),
        y=tokens_local,
        colorscale="YlOrBr",
        name="Attention",
        colorbar=dict(title="attention", x=1.02),
        hovertemplate="step=%{x}<br>token=%{y}<br>attention=%{z:.3f}<extra></extra>",
    ),
    row=2,
    col=1,
)

for action_name in sorted(view_df["action_name"].unique()):
    mask = view_df["action_name"] == action_name
    fig_fin.add_trace(
        go.Scatter(
            x=view_df.index[mask],
            y=view_df["attn_entropy"][mask],
            mode="markers",
            name=f"Action: {action_name}",
            marker=dict(size=8),
            hovertemplate=(
                "step=%{x}<br>action=" + action_name +
                "<br>attn_entropy=%{y:.3f}<extra></extra>"
            ),
            showlegend=False,
        ),
        row=3,
        col=1,
    )

# Regime shading in action panel
for ridx, rname in enumerate(view_df["regime_name"].astype(str).unique()):
    mask = view_df["regime_name"].astype(str) == rname
    if not mask.any():
        continue
    xvals = view_df.index[mask]
    fig_fin.add_trace(
        go.Scatter(
            x=xvals,
            y=[-0.02] * len(xvals),
            mode="markers",
            marker=dict(symbol="square", size=7),
            name=f"Regime: {rname}",
            hovertemplate="step=%{x}<br>regime=" + rname + "<extra></extra>",
            showlegend=(ridx < 4),
        ),
        row=3,
        col=1,
    )

fig_fin.update_layout(
    title="Finance XAI Dashboard: Where the RL policy attends when allocating portfolio",
    height=980,
    width=1400,
    barmode="overlay",
    legend=dict(orientation="h", y=1.03, x=0),
)
fig_fin.update_yaxes(title_text="return", tickformat=".1%", row=1, col=1)
fig_fin.update_yaxes(title_text="token", row=2, col=1)
fig_fin.update_yaxes(title_text="attention entropy", row=3, col=1)
fig_fin.update_xaxes(title_text="test step", row=3, col=1)
fig_fin.show()

# --- Figure 2: Regime vs action preference heatmap ---
pivot = (
    act_local.groupby(["regime_name", "action_name"])  # type: ignore[arg-type]
    .size()
    .reset_index(name="count")
)

regimes = sorted(pivot["regime_name"].unique())
actions = sorted(pivot["action_name"].unique())
mat = np.zeros((len(regimes), len(actions)), dtype=float)

for _, row in pivot.iterrows():
    i = regimes.index(row["regime_name"])
    j = actions.index(row["action_name"])
    mat[i, j] = row["count"]

row_sums = mat.sum(axis=1, keepdims=True) + 1e-12
mat_pct = mat / row_sums

fig_policy = go.Figure(
    data=[
        go.Heatmap(
            z=mat_pct,
            x=actions,
            y=regimes,
            colorscale="Teal",
            hovertemplate="regime=%{y}<br>action=%{x}<br>prob=%{z:.2%}<extra></extra>",
            colorbar=dict(title="P(action|regime)"),
        )
    ]
)
fig_policy.update_layout(
    title="Policy Behavior by Market Regime (Empirical action probabilities)",
    height=420,
    xaxis_title="action",
    yaxis_title="regime",
)
fig_policy.show()

# --- Figure 3: Named finance attention heads (tab-style selector) ---
action_mix = view_df["action_name"].value_counts(normalize=True)

fig_heads = xai_plotly.create_finance_attention_heads_figure(
    sample_attention=attn_local[:show_n],
    sample_saliency=sal_local[:show_n],
    weekly_returns=view_df["weekly_return"].values,
    token_labels=tokens_local,
    action_labels=action_mix.index.tolist(),
    action_distribution=action_mix.values,
    title="Finance Attention Head Explorer",
)
fig_heads.show()

print("Finance explainability view ready:")
print("1) Links attention weights to real portfolio returns over time.")
print("2) Shows which lookback token the policy emphasizes at each step.")
print("3) Summarizes action preferences by inferred market regime.")
print("4) Adds tab-style named heads: Trend-following, Risk-aware, Local context.")